# 03. Experiments

In [30]:
import os, sys, random, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report
import joblib


sys.path.insert(0, '../src')
from preprocessing import get_audio_features
from modeling import (
    split_data, evaluate_model, get_all_models, 
    train_models, hyperparameter_tuning, SEED
)

warnings.filterwarnings('ignore')
np.random.seed(SEED)
random.seed(SEED)

## 1.Подргрузка и разбиение

In [31]:
DATA_PROCESSED = '../data/processed/spotify_processed.csv'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs('../report/images', exist_ok=True)

df = pd.read_csv(DATA_PROCESSED)
audio_features = get_audio_features()
X, y = df[audio_features], df['is_popular']
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

## 2. Обучение

In [32]:
results_df, trained_models = train_models(X_train, X_val, y_train, y_val)
print(results_df[['Model', 'F1', 'Accuracy', 'Precision', 'Recall', 'ROC_AUC']])

                Model        F1  Accuracy  Precision    Recall   ROC_AUC
2        RandomForest  0.684391  0.978480   0.908884  0.548831  0.880025
5        DecisionTree  0.521411  0.955556   0.480836  0.569464  0.796045
3             xGBoost  0.410095  0.967193   0.870536  0.268226  0.867852
1              KNN_k5  0.179904  0.949883   0.295597  0.129298  0.785850
4          GaussianNB  0.134988  0.898070   0.105590  0.187070  0.691501
0  LogisticRegression  0.000000  0.957485   0.000000  0.000000  0.703724


До перебора параметров лучше всего показал себя RandomForest

## 3. GridSearchCV

In [33]:
models_dict = {}
model_results = {}
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

rf_tuned = hyperparameter_tuning(
    X_train, y_train,
    'RandomForest',
    RandomForestClassifier(random_state=SEED, n_jobs=-1)
)
rf_tuned_metrics = evaluate_model(rf_tuned, X_val, y_val)
models_dict['RF_Tuned'] = (rf_tuned, rf_tuned_metrics)
knn_tuned = hyperparameter_tuning(
    X_train, y_train,
    'KNN_k5',
    KNeighborsClassifier()
)
knn_tuned_metrics = evaluate_model(knn_tuned, X_val, y_val)
models_dict['KNN_Tuned'] = (knn_tuned, knn_tuned_metrics)
xgb_tuned = hyperparameter_tuning(
    X_train, y_train,
    'xGBoost',
    XGBClassifier(
        random_state=SEED,
        verbosity=0,
        eval_metric='logloss'
    )
)
xgb_tuned_metrics = evaluate_model(xgb_tuned, X_val, y_val)
models_dict['XGB_Tuned'] = (xgb_tuned, xgb_tuned_metrics)
gaus_tuned = hyperparameter_tuning(
    X_train, y_train,
    'GaussianNB',
    GaussianNB(
    )
)
gaus_tuned_metrics = evaluate_model(gaus_tuned, X_val, y_val)
models_dict['GNB_Tuned'] = (gaus_tuned, gaus_tuned_metrics)
dt_tuned = hyperparameter_tuning(
    X_train, y_train,
    'DecisionTree',
    DecisionTreeClassifier(random_state=SEED)
)
dt_tuned_metrics = evaluate_model(dt_tuned, X_val, y_val)
models_dict['DT_Tuned'] = (dt_tuned, dt_tuned_metrics)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr = LogisticRegression(
    random_state=SEED,
    class_weight='balanced',
    max_iter=1000
)
lr.fit(X_train_scaled, y_train)
lr_metrics = evaluate_model(lr, X_val_scaled, y_val)
models_dict['LR_Tuned'] = (lr, lr_metrics)
for name, (_, metrics) in models_dict.items():
    row = metrics.copy()
    model_results[name] = row
results_df = pd.DataFrame(model_results).T
results_df = results_df[
    ["F1", "Accuracy", "Precision", "Recall", "ROC_AUC"]
].sort_values("F1", ascending=False)
display(results_df)

GridSearch RandomForest:   0%|          | 0/18 [00:00<?, ?comb/s]

GridSearch KNN_k5:   0%|          | 0/8 [00:00<?, ?comb/s]

GridSearch xGBoost:   0%|          | 0/8 [00:00<?, ?comb/s]

GridSearch GaussianNB:   0%|          | 0/2 [00:00<?, ?comb/s]

GridSearch DecisionTree:   0%|          | 0/36 [00:00<?, ?comb/s]

,F1,Accuracy,Precision,Recall,ROC_AUC
RF_Tuned,0.684437,0.978538,0.912844,0.547455,0.872712
KNN_Tuned,0.660641,0.975848,0.820408,0.552957,0.832139
DT_Tuned,0.521411,0.955556,0.480836,0.569464,0.796045
XGB_Tuned,0.450207,0.969006,0.915612,0.298487,0.875978
GNB_Tuned,0.134988,0.898070,0.105590,0.187070,0.691501
LR_Tuned,0.126446,0.562865,0.069093,0.744154,0.702501


## 4. Вывод результов

In [34]:
results_final_dict = {}
for model_name, model in trained_models.items():
    if model_name in ['F1', 'Accuracy', 'Precision', 'Recall', 'ROC_AUC']:
        continue
    metrics = evaluate_model(model, X_val, y_val)
    results_final_dict[model_name] = metrics
for model_name, (model, metrics) in models_dict.items():
    results_final_dict[model_name] = metrics
results_final_dict.update(model_results)

results_df_final = pd.DataFrame(results_final_dict).T
results_df_final = results_df_final.sort_values('F1', ascending=False)
print(results_df_final.to_string())

                          F1  Accuracy  Precision    Recall   ROC_AUC
RF_Tuned            0.684437  0.978538   0.912844  0.547455  0.872712
RandomForest        0.684391  0.978480   0.908884  0.548831  0.880026
KNN_Tuned           0.660641  0.975848   0.820408  0.552957  0.832139
DecisionTree        0.521411  0.955556   0.480836  0.569464  0.796045
DT_Tuned            0.521411  0.955556   0.480836  0.569464  0.796045
XGB_Tuned           0.450207  0.969006   0.915612  0.298487  0.875978
xGBoost             0.410095  0.967193   0.870536  0.268226  0.867852
KNN_k5              0.179904  0.949883   0.295597  0.129298  0.785850
GNB_Tuned           0.134988  0.898070   0.105590  0.187070  0.691501
GaussianNB          0.134988  0.898070   0.105590  0.187070  0.691501
LR_Tuned            0.126446  0.562865   0.069093  0.744154  0.702501
LogisticRegression  0.000000  0.957485   0.000000  0.000000  0.703724


Лучше всех с выявлением сложных нелинейных зависимостей и шумами справился RandomForest. Огромный прирост видно у KNN после подбора параметров. 
DecisionTree можно сказать лучшая среди baseline моделей. XGB имеет самый высокий precision, но достаточно низкий recall, много популярных треков было скипнуто.
GaussianNB и LogisticRegression ожидаемо не справились из-за большого дисбаланса.

## 5. Выводы по лучшей моделе

In [36]:
best_name = results_df_final.index[0]
best_f1 = results_df_final.iloc[0]['F1']

if best_name in models_dict:
    best_model = models_dict[best_name][0]
elif best_name in trained_models:
    best_model = trained_models[best_name]
else:
    best_model = models_dict.get(best_name, (None, None))[0]
    

X_test_scaled = scaler.transform(X_test)
scaled_models = {'LR_Baseline', 'Voting'}
X_test_best = X_test_scaled if best_name in scaled_models else X_test

y_test_pred = best_model.predict(X_test_best)

print(f'\n Лучшая модель: {best_name}')
print(f'F1: {best_f1:.7f}')
test_metrics = evaluate_model(best_model, X_test_best, y_test)
print(classification_report(y_test, y_test_pred))


 Лучшая модель: RF_Tuned
F1: 0.6844368
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     16373
           1       0.91      0.55      0.69       727

    accuracy                           0.98     17100
   macro avg       0.95      0.77      0.84     17100
weighted avg       0.98      0.98      0.98     17100



Видно, что много популярных треков модель все же скипает, но тем не менее у неё с высокой вероятностью у неё выходит определять популярные треки в такой несбалансированной выборке